In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
# NOTEBOOK_DIR == .../STUDY-DATA/third_week/12_24

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]  # .../STUDY-DATA
DATA_DIR = PROJECT_ROOT / "third_week" / "data_csv"

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("tone_vectors exists:", (DATA_DIR / "tone_vectors.pkl").exists())
print("tone_metadata_extended exists:", (DATA_DIR / "tone_metadata_extended.csv").exists())

# 같은 폴더의 .py import 보장 (agent10_rule_engine.py, map_tone_vector_to_params.py)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

Rule Engine + ToneAnalyzer

In [ ]:
import numpy as np
import pandas as pd

# 네 팀원이 만든 import-safe 버전 기준:
from map_tone_vector_to_params import ToneAnalyzer

# 네가 만든 rule engine 파일이 여기에 있다고 가정:
# third_week/12_24/agent10_rule_engine.py
from agent10_rule_engine import generate_message

ToneAnalyzer 인스턴스 생성

In [ ]:
PATH_CENTROIDS = DATA_DIR / "tone_vectors.pkl"
PATH_META = DATA_DIR / "tone_metadata_extended.csv"

analyzer = ToneAnalyzer.from_files(PATH_CENTROIDS, PATH_META)

print("loaded tone ids:", list(analyzer.tone_centroids.keys()))
print("meta columns:", analyzer.tone_meta.columns.tolist())

In [ ]:
테스트 입력 준비 (persona / product_meta / slot_schema)

In [ ]:
persona = "민감성 피부 직장인"

product_meta = {
    "name": "라네즈 워터뱅크 블루 히알루로닉 크림",
    "key_benefit": "보습 장벽 강화",
    "proof_point": "임상 데이터",
    "emotion_phrase": "편안한 일상"
}

slot_schema = ["intro", "proof", "emotion", "cta"]

tone_vector 준비 (실전용 / 더미용)
(A) 실전: tone_vector가 이미 있으면 그대로 넣기
# tone_vector = 실제 임베딩 벡터 (np.ndarray shape (D,))
# 예: tone_vector = your_vector

(B) 더미 테스트: centroid 평균으로 “그럴듯한 입력” 만들기

In [ ]:
# centroid 중 하나를 골라서 약간 노이즈
tone_id_for_test = list(analyzer.tone_centroids.keys())[0]
base = analyzer.tone_centroids[tone_id_for_test]
noise = np.random.normal(0, 0.01, size=base.shape)

tone_vector = (base + noise).astype(float)

print("test tone_id base:", tone_id_for_test)
print("tone_vector shape:", tone_vector.shape)

map_tone_vector_to_params 단독 검증

In [ ]:
params = analyzer.map_tone_vector_to_params(tone_vector)
params

케이스 1) generate_message(persona, tone_vector, slot_schema, product_meta) 형태라면

In [ ]:
result = generate_message(
    persona=persona,
    tone_vector=tone_vector,
    slot_schema=slot_schema,
    product=product_meta  
)
result

케이스 2) generate_message가 analyzer/params를 외부 주입받도록 되어있다면 (예: params를 넘기는 구조)

In [ ]:
# params = analyzer.map_tone_vector_to_params(tone_vector)
# result = generate_message(persona, params, slot_schema, product_meta)
# result

In [ ]:
from pprint import pprint

print("\n--- MESSAGE ---")
print(result["message"])

print("\n--- TRACE ---")
pprint(result["trace"])

trace 슬롯 단위 확인

In [ ]:
trace_slots = result.get("trace", {}).get("slots", {})
trace_slots

재현성 체크

In [ ]:
result2 = generate_message(
    persona=persona,
    tone_vector=tone_vector,
    slot_schema=slot_schema,
    product=product_meta
)

print(result["message"] == result2["message"])
print(result["trace"] == result2["trace"])